# **Imports**

In [1]:
# Imports
from pathlib import Path
import json
import re
import os
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from jiwer import cer, wer

In [2]:
# Imports for model
import torch
import torchaudio
import torchaudio.transforms as T
import torch.nn as nn
from tokenizer import TokenizerBPE01
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import pickle as pkl
import json

# CTC
from ctc_utils import *

# Models
from models import models as md

from datasets import load_dataset

d:\workplace\Project\parts\STT\ctc_ver01\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# **Models comparison**

In [3]:
ROOT_DIR = Path(os.getcwd())

VOCAB_DIRS = {
    "vocab (70 tokens)": ROOT_DIR / "trained_models/vocab",
    #"vocab1 (765 tokens)": ROOT_DIR / "trained_models/vocab1",
}

MODEL_INFO_JSON = ROOT_DIR / "models_config.json"

champions = {
    "vocab (70 tokens)": 24,  # Selected from plot below
    #"vocab1 (765 tokens)": 25,
}

TOP_N = 5

In [4]:
# Get models info
def load_model_info(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        return json.load(f)

def format_model_name(model_idx, model_info):
    info = model_info[str(model_idx)]

    model_type = info["model_type"]
    model_no = info["model_no"]
    params = info.get("model_parameters", {})

    if params:
        param_string = ", ".join(
            f"{key}={value}"
            for key, value in params.items()
        )
        return f"{model_no}: {model_type} ({param_string})"

    return f"{model_no}: {model_type}"


# get results files
def find_result_files(vocab_dir):
    results = []

    pattern = re.compile(r"results_model(\d+)\.csv$", re.IGNORECASE)

    for csv_path in vocab_dir.rglob("results_model*.csv"):
        match = pattern.match(csv_path.name)
        if match is None:
            continue

        model_idx = int(match.group(1))

        results.append({
            "model_idx": model_idx,
            "path": csv_path
        })

    return sorted(results, key=lambda x: x["model_idx"])


# Get full model result info
def full_model_info(vocab_dir, model_info):
    models = {}

    files = find_result_files(vocab_dir)

    for item in files:
        model_idx = item["model_idx"]
        df = pd.read_csv(item["path"])

        models[model_idx] = {
            "name": format_model_name(model_idx, model_info),
            "path": item["path"],
            "data": df
        }
    return models

In [5]:
# Plotting graphs
def plot_vocab_models(vocab, models, initial_visible):
    fig = go.Figure()

    for model_idx, model in models.items():
        visible = model_idx in initial_visible
        df = model["data"]
        name = model["name"]

        # Train
        fig.add_trace(
            go.Scatter(
                x=df["Epoch"],
                y=df["Train"],
                mode="lines+markers",
                name=f"{name} - Train",
                legendgroup=str(model_idx),
                hovertemplate=(
                    f"{name}<br>"
                    "Epoch: %{x}<br>"
                    "Train: %{y}<extra></extra>"
                ),
                visible=visible
                
            )
        )


       # Validation
        fig.add_trace(
            go.Scatter(
                x=df["Epoch"],
                y=df["Validation"],
                mode="lines+markers",
                name=f"{name} - Validation",
                legendgroup=str(model_idx),
                hovertemplate=(
                    f"{name}<br>"
                    "Epoch: %{x}<br>"
                    "Train: %{y}<extra></extra>"
                ),
                visible=visible
            )
        )

    fig.update_layout(
        title=f"Models comparison - {vocab}",
        xaxis_title="Epoch",
        yaxis_title="Score",
        hovermode="x unified",
        template="plotly_white",
        legend_title="Model",

        width=1200,
        height=800
    )

    fig.show()


def plot_champion_models(champion_models):
    fig = go.Figure()

    for vocab_name, model_idx, model in champion_models:
        df = model["data"]

        fig.add_trace(
            go.Scatter(
                x=df["Epoch"],
                y=df["Train"],
                mode="lines+markers",
                name=f"{vocab_name} - Train"
            )
        )

        fig.add_trace(
            go.Scatter(
                x=df["Epoch"],
                y=df["Validation"],
                mode="lines+markers",
                name=f"{vocab_name} - Validation"
            )
        )


    fig.update_layout(
        title="Final champion models comparison",
        xaxis_title="Epoch",
        yaxis_title="Score",
        hovermode="x unified",
        template="plotly_white",
        legend_title="Model",
        width=1200,
        height=800
    )

    fig.show()
    


In [6]:
model_info = load_model_info(MODEL_INFO_JSON)
all_vocab_models = {}

for vocab_name, vocab_dir in VOCAB_DIRS.items():
    print(f"[{vocab_name}]")

    if not vocab_dir.exists():
        print(f"Directory {vocab_dir} does not exist.")
        continue

    models = full_model_info(vocab_dir, model_info)
    all_vocab_models[vocab_name] = models

    # Get final validation score for each model
    final_scores = []

    for model_idx, model in models.items():
        df = model["data"]

        final_validation = df["Validation"].iloc[-1]

        final_scores.append({
            "model_idx": model_idx,
            "score": final_validation
        })

    # Lowest validation loss first
    final_scores.sort(key=lambda x: x["score"])

    # Keep only the top N visible initially
    initial_visible = {
        x["model_idx"]
        for x in final_scores[:TOP_N]
    }

    print(f"Found {len(models)} models")


    if models:
        plot_vocab_models(vocab_name, models, initial_visible)

    print()


[vocab (70 tokens)]
Found 26 models


Best model: model 24  
Early epoch: learn rapidly with the steepest loss line before stabilizing at epoch 6.

In [7]:
# Compare between two vocab's champion model.
champion_models = []

for vocab_name, model_idx in champions.items():

    if model_idx is None:
        print(f"No champion model selected for {vocab_name}")
        continue

    models = all_vocab_models.get(vocab_name, {})

    champion_models.append((vocab_name, model_idx, models[model_idx]))

    if len(champion_models) == 2:
        plot_champion_models(champion_models)

*For comparison between two vocab size set of models.*  
Eventhough vocab size 70 model show lower ctc_loss, ctc loss is affected by vocab size - lower chance to select the correct token per step, and tokenized sequence length reduced by using tokens represent 2 or more characters. ctc loss is not reasonable for cross-vocabulary comparison.

# **Champion model selection**

In [8]:
champions_model_1 = ROOT_DIR / "trained_models/vocab/deep_bilstm/model24.pt"
#champions_model_2 = ROOT_DIR / "trained_models/vocab1/bilstm_projection/model25.pt"

vocab_path = ROOT_DIR / "vocab.json"
vocab1_path = ROOT_DIR / "vocab1.json"

results = {
    "vocab 40 tokens - model24": {
        "wer": [],
        "cer": []
    },

    # "vocab 765 tokens - model25": {
    #         "wer": [],
    #         "cer": []
    #     }
}



In [9]:
# Loading validation data
train_set = load_dataset(
    "openslr/librispeech_asr",
    "clean",
    split="train.100"
)

val_set = load_dataset(
    "openslr/librispeech_asr",
    "clean",
    split="validation"
)
print()


# Device
#device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = torch.device("cpu")
print("Device:", device)
print()

'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/datasets/openslr/librispeech_asr/resolve/71cacbfb7e2354c4226d01e70d77d5fca3d04ba1/librispeech_asr.py
Retrying in 1s [Retry 1/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/datasets/openslr/librispeech_asr/resolve/71cacbfb7e2354c4226d01e70d77d5fca3d04ba1/librispeech_asr.py
Retrying in 2s [Retry 2/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/datasets/openslr/librispeech_asr/resolve/71cacbfb7e2354c4226d01e70d77d5fca3d04ba1/librispeech_asr.py
Retrying in 4s [Retry 3/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/datasets/openslr/librispeech_asr/resolve/71cacbfb7e2354c4226d01e70d77d5fca3d04ba1/librispeech_asr.py
Retrying in 8s [Retry 4/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/datasets/openslr/librispeech_asr/resolve/71cacbfb


Device: cpu



In [10]:
add_blank = ['<Blank>']

tokenizer = TokenizerBPE01()
tokenizer.loadFromJSON(vocab_path)
tokenizer.update(add_blank)
blank_id = tokenizer.vocab['<Blank>']
pad_id = len(tokenizer.vocab)



Initiate vocabulary of length: 38
Loaded.
Added 1/1 into vocabulary.
Existed: 0/1.
Current vocabulary length: 39


In [11]:
mfcc_transform = torchaudio.transforms.MFCC(
    sample_rate=16000,
    n_mfcc=13,
    melkwargs={
        "n_fft": 1200,
        "win_length": 1200, # 75ms timeframe
        "hop_length": 600,  
        "n_mels": 40
    }
    )

In [12]:
# Data preparation
# Normalizer
class Normalize:
    def __init__(self, mean, std):
        self.mean = mean
        self.std = std

    def __call__(self, x):
        return (x - self.mean) / (self.std + 1e-8)

    def to(self, device):
        self.mean = self.mean.to(device)
        self.std = self.std.to(device)

with open(ROOT_DIR / "normalizer1.pkl", "rb") as f:
    normalizer = pkl.load(f)
    normalizer.to(device) 


def get_input_and_target(dataset):
    # Input output
    input_audios = dataset['audio']
    target_sentences = dataset['text']

    return input_audios, target_sentences


def preprocess_audio(audio, mfcc_transform, device):
    try:
        input_audio = torch.tensor(audio)
    except:
        # For codec-style input
        input_audio = torch.tensor(audio["array"])

    mfccs = mfcc_transform(input_audio).to(device).transpose(0, 1)

    seq_len = len(mfccs)

    return mfccs, seq_len

In [13]:
audios, targets = get_input_and_target(val_set)
del val_set

In [14]:
# Models loading
# Model 1
with open('models_config.json', 'r') as file:
    cfg = json.load(file)

model = cfg[str(24)]
model_no = model["model_no"]
model_type = model["model_type"]
model_parameters = model["model_parameters"]

model1 = md[model_type](
    input_size=13,
    output_size=len(tokenizer.vocab),
    **model_parameters
).to(device)

model1.load_state_dict(
    torch.load(champions_model_1, map_location=device)
)


# # Model 2
# model = cfg[str(25)]
# model_no = model["model_no"]
# model_type = model["model_type"]
# model_parameters = model["model_parameters"]

# model2 = models[model_type](
#     input_size=13,
#     output_size=len(tokenizer1.vocab),
#     **model_parameters
# ).to(device)

# model2.load_state_dict(
#     torch.load(champions_model_2, map_location=device)
# )

<All keys matched successfully>

In [15]:
for audio, target in zip(audios, targets):

    # Preprocess
    mfccs = preprocess_audio(audio, mfcc_transform, device)[0]
    mfccs = mfccs.unsqueeze(0)

    # Inference
    with torch.no_grad():
        logits1 = model1(normalizer(mfccs))
        output1, _ = beam_search(logits1, 10, blank_id)

        # logits2 = model2(normalizer(mfccs))
        # output2, _ = beam_search(logits2, 10, blank_id1)


    # Decode
    prediction = tokenizer.tokenToWords(collpase(output1[0], blank_id))
    #prediction1 = tokenizer1.tokenToWords(collpase(output2[0], blank_id1))

    # Metrics
    wer1 = wer(target.lower(), prediction)
    cer1 = cer(target.lower(), prediction)

    # wer2 = wer(target, prediction1)
    # cer2 = cer(target, prediction1)

    # Save
    results["vocab 40 tokens - model24"]["wer"].append(wer1)
    results["vocab 40 tokens - model24"]["cer"].append(cer1)

    # results["vocab 765 tokens - model25"]["wer"].append(wer2)
    # results["vocab 765 tokens - model25"]["cer"].append(cer2)

print("Example target:")
print(target.lower())

print("Prediction")
print(prediction)

Example target:
i could not love thee dear so much loved i not honor more
Prediction
i could not eve eear so lops fo lelv ti nowt al her more


In [16]:
def make_report(results):
    report = []

    for model_name, metrics in results.items():

        wer = np.array(metrics["wer"], dtype=float)
        cer = np.array(metrics["cer"], dtype=float)

        report.append({
            "Model": model_name,

            "Samples": len(wer),

            "WER Mean": wer.mean(),
            "WER Median": np.median(wer),
            "WER Std": wer.std(),
            "WER P25": np.percentile(wer, 25),
            "WER P75": np.percentile(wer, 75),
            "WER Min": wer.min(),
            "WER Max": wer.max(),

            "CER Mean": cer.mean(),
            "CER Median": np.median(cer),
            "CER Std": cer.std(),
            "CER P25": np.percentile(cer, 25),
            "CER P75": np.percentile(cer, 75),
            "CER Min": cer.min(),
            "CER Max": cer.max(),
        })

    return pd.DataFrame(report)


report = make_report(results)

print(report.to_string(index=False))

                    Model  Samples  WER Mean  WER Median  WER Std  WER P25  WER P75  WER Min  WER Max  CER Mean  CER Median  CER Std  CER P25  CER P75  CER Min  CER Max
vocab 40 tokens - model24     2703  0.802369         0.8  0.18791 0.692308 0.894737 0.166667 2.333333  0.335365    0.329114 0.096573 0.269269 0.393272 0.035714 0.869565
